# Project Homework: Applying GitHub Ingestion to OWASP LLM Top 10

This notebook applies the `read_repo_data()` pattern learned in the course to a different repository structure. The OWASP LLM Top 10 repository provides security documentation for large language model applications.

## What's Different

Unlike the course repositories (DataTalks FAQ and Evidently docs):
- **OWASP has NO frontmatter** - files are plain markdown without YAML metadata blocks
- **Different organization** - nested structure with 2_0_vulns/, documentation/, translations/ folders
- **More files** - 542 markdown documents vs 95-1285 in course repos

## Key Learning

The same `read_repo_data()` implementation works across all repositories because:
1. python-frontmatter gracefully handles files without frontmatter (returns empty metadata dict)
2. In-memory zip processing works regardless of repository structure
3. Markdown filtering works uniformly across file organizations

In [1]:
# Import required libraries
import requests
from zipfile import ZipFile
from io import BytesIO
import frontmatter
from pathlib import Path
from typing import List, Dict, Any

## Helper Functions

These functions are copied from course/day1.ipynb (per learning objective - adaptation through reuse, not reinvention). Each function has been enhanced with type hints and docstrings per project/ engineering standards.

In [2]:
def download_repo_zip(repo_owner: str, repo_name: str) -> bytes:
    """Download GitHub repository as zip archive.

    Uses GitHub's codeload API which provides repositories as zip archives
    without requiring authentication for public repos.

    Args:
        repo_owner: GitHub username or organization (e.g., 'OWASP')
        repo_name: Repository name (e.g., 'www-project-top-10-for-large-language-model-applications')

    Returns:
        Binary content of zip archive

    Raises:
        requests.HTTPError: If download fails (repo not found, network error, etc.)
    """
    url = f"https://codeload.github.com/{repo_owner}/{repo_name}/zip/refs/heads/main"
    print(f"Downloading {repo_owner}/{repo_name} from {url}")

    response = requests.get(url, timeout=30)
    response.raise_for_status()

    print(f"Downloaded {len(response.content)} bytes")
    return response.content

In [3]:
def is_markdown_file(filename: str) -> bool:
    """Check if file is a markdown document (.md or .mdx).

    Args:
        filename: Full path to file in zip archive

    Returns:
        True if file ends with .md or .mdx, False otherwise
    """
    path = Path(filename)
    return path.suffix in ['.md', '.mdx']

In [4]:
def parse_markdown_with_frontmatter(content_bytes: bytes) -> Dict[str, Any]:
    """Parse markdown file extracting frontmatter metadata and content.

    Handles files without frontmatter gracefully - returns empty metadata dict.
    This is critical for OWASP repository which has no frontmatter.

    Args:
        content_bytes: Raw file content as bytes

    Returns:
        Dictionary with:
        - 'metadata': dict of frontmatter key-value pairs (empty {} if no frontmatter)
        - 'content': str of markdown content after frontmatter (or full content if none)
    """
    content_str = content_bytes.decode('utf-8', errors='ignore')
    post = frontmatter.loads(content_str)

    return {
        'metadata': dict(post.metadata),  # Empty dict {} if no frontmatter
        'content': post.content
    }

## Main Function: read_repo_data()

This is the same orchestration logic from the course, proving that good abstractions work across different repository structures.

In [5]:
def read_repo_data(repo_owner: str, repo_name: str) -> List[Dict[str, Any]]:
    """Download GitHub repository and extract markdown documentation with metadata.

    This implementation is identical to course/day1.ipynb because the core
    pattern is universal. The interesting part is how it handles repositories
    with different structures and frontmatter conventions.

    Args:
        repo_owner: GitHub username or organization (e.g., 'OWASP')
        repo_name: Repository name

    Returns:
        List of dictionaries, each containing:
        - 'filename': str - path to file within repository
        - 'metadata': dict - YAML frontmatter key-value pairs (empty {} if no frontmatter)
        - 'content': str - markdown content after frontmatter (or full content if none)
    """
    # Download repository as zip
    zip_content = download_repo_zip(repo_owner, repo_name)

    # Open zip archive in memory
    with ZipFile(BytesIO(zip_content)) as zip_file:
        # Get all files
        all_files = zip_file.namelist()
        print(f"Total files in archive: {len(all_files)}")

        # Filter for markdown files
        markdown_files = [f for f in all_files if is_markdown_file(f)]
        print(f"Markdown files found: {len(markdown_files)}")

        # Process each markdown file
        documents = []
        for filename in markdown_files:
            file_content = zip_file.read(filename)
            parsed = parse_markdown_with_frontmatter(file_content)

            doc = {
                'filename': filename,
                'metadata': parsed['metadata'],
                'content': parsed['content']
            }
            documents.append(doc)

        print(f"Successfully processed {len(documents)} documents")
        return documents

## Testing with OWASP LLM Top 10 Repository

Repository: https://github.com/OWASP/www-project-top-10-for-large-language-model-applications

This repository documents the top 10 security risks for LLM applications. It has a different structure than our course examples and uses plain markdown (no frontmatter).

In [6]:
# Download and process OWASP repository
docs = read_repo_data("OWASP", "www-project-top-10-for-large-language-model-applications")

Downloaded 320788136 bytes
Total files in archive: 1076
Markdown files found: 542
Successfully processed 542 documents


In [7]:
# Display repository characteristics
print(f"OWASP LLM Top 10 Repository Analysis")
print(f"=" * 80)
print(f"Total documents: {len(docs)}")
print(f"Documents with frontmatter: {sum(1 for d in docs if d['metadata'])}")
print(f"Documents without frontmatter: {sum(1 for d in docs if not d['metadata'])}")
print()

# Show sample documents
print(f"Sample documents (first 3):")
for i, doc in enumerate(docs[:3], 1):
    print(f"\n{i}. {doc['filename']}")

    # Handle empty metadata gracefully
    if doc['metadata']:
        print(f"   Metadata: {doc['metadata']}")
    else:
        print(f"   Metadata: None (no frontmatter)")

    # Show content preview
    content_preview = doc['content'][:150].replace('\n', ' ')
    print(f"   Content preview: {content_preview}...")
    print(f"   Total length: {len(doc['content'])} characters")

# Compare to course repos
print(f"\n" + "=" * 80)
print(f"Comparison with Course Repositories:")
print(f"  DataTalks FAQ:     1,285 files, most with frontmatter")
print(f"  Evidently docs:       95 files, most with frontmatter")
print(f"  OWASP LLM Top 10:   {len(docs)} files, NO frontmatter")
print(f"\nKey insight: The same code works across all repos because python-frontmatter")
print(f"gracefully handles files without frontmatter (returns empty metadata dict).")

OWASP LLM Top 10 Repository Analysis
Total documents: 542
Documents with frontmatter: 3
Documents without frontmatter: 539

Sample documents (first 3):

1. www-project-top-10-for-large-language-model-applications-main/.github/PULL_REQUEST_TEMPLATE.md
   Metadata: None (no frontmatter)
   Content preview: # [Title of Your PR]  **Key Changes:**  - [ ] List major changes and core updates - [ ] Keep each line under 80 characters - [ ] Focus on the "what" a...
   Total length: 569 characters

2. www-project-top-10-for-large-language-model-applications-main/2_0_vulns/LLM00_Preface.md
   Metadata: None (no frontmatter)
   Content preview: ## Letter from the Project Leads  The OWASP Top 10 for Large Language Model Applications started in 2023 as a community-driven effort to highlight and...
   Total length: 3167 characters

3. www-project-top-10-for-large-language-model-applications-main/2_0_vulns/LLM01_PromptInjection.md
   Metadata: None (no frontmatter)
   Content preview: ## LLM01:2025 Pro

## What I Learned

**Repository Structure Adaptability:**
- The OWASP repository has 542 markdown files across nested directories (2_0_vulns/, documentation/, resources/)
- Unlike course repos that use frontmatter for metadata, OWASP files are plain markdown
- The `read_repo_data()` implementation handles this gracefully - no code changes needed

**Frontmatter Handling:**
- python-frontmatter library returns empty `metadata: {}` for files without frontmatter
- This makes the implementation robust - works with or without frontmatter
- Important to check `if doc['metadata']:` before displaying metadata fields to avoid confusing output

**Why This Matters for RAG:**
- Some documentation uses frontmatter (Jekyll, Hugo, Next.js sites) for structured metadata
- Other documentation is plain markdown with all structure in headers
- A good RAG ingestion pipeline must handle both cases
- Day 2 chunking will need to adapt strategy based on whether frontmatter provides structure

___

## Day 2: Chunking

Apply chunking strategies to OWASP documentation from Day 1. We'll implement sliding window chunking as the baseline, with full type annotations and validation.

In [8]:
# Day 2 imports
import copy
import tiktoken
import re
from typing import Any

In [9]:
def count_tokens(text: str, encoding: str = "cl100k_base") -> int:
    """Count tokens in text using tiktoken.

    Args:
        text: Text to count tokens for.
        encoding: Tiktoken encoding name. Use 'cl100k_base' for GPT-4/3.5-turbo
            and text-embedding-ada-002.

    Returns:
        Number of tokens in the text.

    Example:
        >>> count_tokens("Hello, world!")
        4
    """
    enc = tiktoken.get_encoding(encoding)
    return len(enc.encode(text))

In [10]:
def chunk_sliding_window(
    doc: dict[str, Any],
    chunk_size: int = 2000,
    overlap: int = 1000,
) -> list[dict[str, Any]]:
    """Split document content into overlapping chunks using sliding window.

    Implements a simple fixed-size sliding window algorithm. Each chunk overlaps
    with the next by `overlap` characters to preserve context at boundaries.

    Args:
        doc: Document dict with required keys:
            - 'content' (str): Document text to chunk
            - 'filename' (str): Source document identifier
            - 'metadata' (dict, optional): Frontmatter metadata to preserve
        chunk_size: Characters per chunk. Default 2000 (~512 tokens).
        overlap: Characters overlap between adjacent chunks. Default 1000.

    Returns:
        List of chunk dicts, each containing:
            - 'filename': Inherited from source document
            - 'metadata': Deep copy of source metadata
            - 'content': Chunk text
            - 'chunk_id': '{filename}-chunk-{index}' identifier
            - 'chunk_index': Zero-based position in chunk sequence
            - 'total_chunks': Total chunks from this document
            - 'chunk_method': 'sliding_window'

    Raises:
        ValueError: If doc missing required keys, or chunk_size <= overlap,
            or overlap < 0.

    Example:
        >>> doc = {'filename': 'test.md', 'metadata': {}, 'content': 'x' * 3000}
        >>> chunks = chunk_sliding_window(doc, chunk_size=2000, overlap=1000)
        >>> len(chunks)
        2
        >>> chunks[0]['chunk_id']
        'test.md-chunk-0'
    """
    # Input validation
    if "content" not in doc or "filename" not in doc:
        raise ValueError("Document must have 'content' and 'filename' keys")
    if chunk_size <= overlap:
        raise ValueError("chunk_size must be greater than overlap")
    if overlap < 0:
        raise ValueError("overlap must be non-negative")

    content = doc["content"]
    chunks: list[dict[str, Any]] = []
    start = 0

    while start < len(content):
        end = start + chunk_size
        chunk_content = content[start:end]

        chunk = {
            "filename": doc["filename"],
            "metadata": copy.deepcopy(doc.get("metadata", {})),
            "content": chunk_content,
            "chunk_id": f"{doc['filename']}-chunk-{len(chunks)}",
            "chunk_index": len(chunks),
            "total_chunks": -1,  # Placeholder, set after loop
            "chunk_method": "sliding_window",
        }
        chunks.append(chunk)
        start += chunk_size - overlap

    # Update total_chunks for all chunks
    total = len(chunks)
    for chunk in chunks:
        chunk["total_chunks"] = total

    return chunks

### Testing on OWASP Documents

Apply sliding window chunking to all OWASP documents from Day 1 and analyze the results.

In [11]:
# Chunk all OWASP documents
sliding_window_chunks: list[dict[str, Any]] = []
for doc in docs:
    chunks = chunk_sliding_window(doc, chunk_size=2000, overlap=1000)
    sliding_window_chunks.extend(chunks)

print(f"Total documents: {len(docs)}")
print(f"Total chunks: {len(sliding_window_chunks)}")
print(f"Average chunks per doc: {len(sliding_window_chunks) / len(docs):.1f}")

Total documents: 542
Total chunks: 3563
Average chunks per doc: 6.6


In [12]:
# Analyze first 3 chunks
for chunk in sliding_window_chunks[:3]:
    char_count = len(chunk["content"])
    token_count = count_tokens(chunk["content"])
    print(f"Chunk: {chunk['chunk_id']}")
    print(f"  Size: {char_count:,} chars, {token_count} tokens")
    print(f"  Token/char ratio: {token_count / char_count:.3f}")
    print(f"  Metadata keys: {list(chunk['metadata'].keys())}")
    print(f"  Position: {chunk['chunk_index'] + 1} of {chunk['total_chunks']}")
    print()

Chunk: www-project-top-10-for-large-language-model-applications-main/.github/PULL_REQUEST_TEMPLATE.md-chunk-0
  Size: 569 chars, 140 tokens
  Token/char ratio: 0.246
  Metadata keys: []
  Position: 1 of 1

Chunk: www-project-top-10-for-large-language-model-applications-main/2_0_vulns/LLM00_Preface.md-chunk-0
  Size: 2,000 chars, 369 tokens
  Token/char ratio: 0.184
  Metadata keys: []
  Position: 1 of 4

Chunk: www-project-top-10-for-large-language-model-applications-main/2_0_vulns/LLM00_Preface.md-chunk-1
  Size: 2,000 chars, 390 tokens
  Token/char ratio: 0.195
  Metadata keys: []
  Position: 2 of 4



In [13]:
# Verify metadata preservation and deep copy
if docs and sliding_window_chunks:
    original_doc = docs[0]
    first_chunk = sliding_window_chunks[0]

    # Check metadata preserved
    assert first_chunk["filename"] == original_doc["filename"], "Filename not preserved"
    assert first_chunk["metadata"] == original_doc.get("metadata", {}), "Metadata not preserved"

    # Check deep copy (modifying chunk doesn't affect original)
    first_chunk["metadata"]["_test_field"] = "test"
    assert "_test_field" not in original_doc.get("metadata", {}), "Deep copy failed"
    del first_chunk["metadata"]["_test_field"]

    print("Metadata preservation verified:")
    print(f"  - Filename preserved: {first_chunk['filename']}")
    print(f"  - Metadata deep copied (not shared reference)")
    print(f"  - Chunk fields added: chunk_id, chunk_index, total_chunks, chunk_method")

Metadata preservation verified:
  - Filename preserved: www-project-top-10-for-large-language-model-applications-main/.github/PULL_REQUEST_TEMPLATE.md
  - Metadata deep copied (not shared reference)
  - Chunk fields added: chunk_id, chunk_index, total_chunks, chunk_method


### Observations

*Document your analysis of the chunking results here:*

- How many chunks were created?
- What's the token/character ratio for OWASP content?
- Are there documents that produce many more chunks than average?
- Does the metadata preservation work correctly?

### Paragraph-Based Chunking

Paragraph chunking splits on blank lines, keeping each paragraph as an atomic unit. This respects natural writing structure but produces variable-sized chunks.

In [14]:
def chunk_by_paragraph(doc: dict[str, Any]) -> list[dict[str, Any]]:
    """Split document into paragraph chunks.

    Splits on blank lines (\\n\\s*\\n pattern), filters empty paragraphs,
    and preserves all metadata from the source document.

    Args:
        doc: Document dict with 'content', 'filename', and optional 'metadata' keys.

    Returns:
        List of chunk dicts, each containing:
        - filename: Inherited from source document
        - metadata: Deep copy of source metadata
        - content: Paragraph text (trimmed)
        - chunk_id: Format '{filename}-chunk-{index}'
        - chunk_index: Position in sequence (0-indexed)
        - total_chunks: Total paragraphs in document
        - chunk_method: 'paragraph'

    Raises:
        KeyError: If doc missing 'content' or 'filename' keys.

    Example:
        >>> doc = {'filename': 'test.md', 'metadata': {}, 'content': 'Para 1.\\n\\nPara 2.'}
        >>> chunks = chunk_by_paragraph(doc)
        >>> len(chunks)
        2
        >>> chunks[0]['chunk_method']
        'paragraph'
    """
    # Split on \n\s*\n pattern - matches paragraph boundaries
    paragraphs = re.split(r'\n\s*\n', doc['content'])

    # Filter empty and trim whitespace
    paragraphs = [p.strip() for p in paragraphs if p.strip()]

    # Create chunks with metadata preservation
    chunks = []
    for i, para_text in enumerate(paragraphs):
        chunk = {
            'filename': doc['filename'],
            'metadata': copy.deepcopy(doc.get('metadata', {})),
            'content': para_text,
            'chunk_id': f"{doc['filename']}-chunk-{i}",
            'chunk_index': i,
            'total_chunks': len(paragraphs),
            'chunk_method': 'paragraph'
        }
        chunks.append(chunk)

    return chunks

### Hybrid Paragraph + Sliding Window Chunking

Combines paragraph and sliding window strategies: splits on paragraph boundaries first, then applies sliding window to paragraphs exceeding the size threshold. This respects natural text boundaries while preventing huge chunks.

In [15]:
def chunk_paragraph_with_sliding_window(
    doc: dict[str, Any],
    max_paragraph_size: int = 2000,
    chunk_size: int = 2000,
    overlap: int = 1000,
) -> list[dict[str, Any]]:
    """Split document by paragraphs with sliding window for oversized paragraphs.

    Combines paragraph and sliding window strategies: splits on paragraph boundaries
    first, then applies sliding window to any paragraph exceeding max_paragraph_size.
    This respects natural text boundaries while preventing huge chunks.

    Args:
        doc: Document dict with required keys:
            - 'content' (str): Document text to chunk
            - 'filename' (str): Source document identifier
            - 'metadata' (dict, optional): Frontmatter metadata to preserve
        max_paragraph_size: Maximum chars before applying sliding window. Default 2000.
        chunk_size: Characters per sliding window chunk. Default 2000.
        overlap: Characters overlap for sliding window chunks. Default 1000.

    Returns:
        List of chunk dicts, each containing:
            - 'filename': Inherited from source document
            - 'metadata': Deep copy of source metadata
            - 'content': Chunk text
            - 'chunk_id': '{filename}-chunk-{index}' identifier
            - 'chunk_index': Zero-based position in chunk sequence
            - 'total_chunks': Total chunks from this document
            - 'chunk_method': 'paragraph_sliding_window'

    Raises:
        ValueError: If doc missing required keys, or invalid parameters.

    Example:
        >>> doc = {'filename': 'test.md', 'metadata': {}, 'content': 'Short.\\n\\nLong ' + 'x' * 3000}
        >>> chunks = chunk_paragraph_with_sliding_window(doc, max_paragraph_size=2000)
        >>> chunks[0]['chunk_method']
        'paragraph_sliding_window'
    """
    # Input validation
    if "content" not in doc or "filename" not in doc:
        raise ValueError("Document must have 'content' and 'filename' keys")
    if max_paragraph_size <= 0:
        raise ValueError("max_paragraph_size must be positive")
    if chunk_size <= overlap:
        raise ValueError("chunk_size must be greater than overlap")
    if overlap < 0:
        raise ValueError("overlap must be non-negative")

    # Step 1: Split by paragraph boundaries (reuse paragraph logic)
    paragraphs = re.split(r'\n\s*\n', doc['content'])
    paragraphs = [p.strip() for p in paragraphs if p.strip()]

    chunks: list[dict[str, Any]] = []

    # Step 2: Process each paragraph
    for para in paragraphs:
        if len(para) <= max_paragraph_size:
            # Small paragraph: keep as-is
            chunk = {
                'filename': doc['filename'],
                'metadata': copy.deepcopy(doc.get('metadata', {})),
                'content': para,
                'chunk_id': f"{doc['filename']}-chunk-{len(chunks)}",
                'chunk_index': len(chunks),
                'total_chunks': -1,  # Updated after loop
                'chunk_method': 'paragraph_sliding_window'
            }
            chunks.append(chunk)
        else:
            # Large paragraph: apply sliding window to THIS paragraph only
            start = 0
            while start < len(para):
                end = start + chunk_size
                chunk_content = para[start:end]

                chunk = {
                    'filename': doc['filename'],
                    'metadata': copy.deepcopy(doc.get('metadata', {})),
                    'content': chunk_content,
                    'chunk_id': f"{doc['filename']}-chunk-{len(chunks)}",
                    'chunk_index': len(chunks),
                    'total_chunks': -1,
                    'chunk_method': 'paragraph_sliding_window'
                }
                chunks.append(chunk)
                start += (chunk_size - overlap)

    # Update total_chunks for all
    total = len(chunks)
    for chunk in chunks:
        chunk['total_chunks'] = total

    return chunks

In [16]:
# Apply hybrid chunking to all OWASP docs
hybrid_chunks: list[dict[str, Any]] = []
para_count = 0
triggered_count = 0

for doc in docs:
    # Count paragraphs before chunking
    paragraphs = re.split(r'\n\s*\n', doc['content'])
    paragraphs = [p.strip() for p in paragraphs if p.strip()]
    para_count += len(paragraphs)
    triggered_count += sum(1 for p in paragraphs if len(p) > 2000)

    chunks = chunk_paragraph_with_sliding_window(doc, max_paragraph_size=2000)
    hybrid_chunks.extend(chunks)

print(f"Hybrid paragraph + sliding window: {len(docs)} docs -> {len(hybrid_chunks)} chunks")
print(f"Average: {len(hybrid_chunks) / len(docs):.1f} chunks per doc")
print()
print(f"Paragraphs analyzed: {para_count}")
print(f"Paragraphs exceeding threshold (triggered sliding window): {triggered_count}")
print(f"Trigger rate: {(triggered_count / para_count) * 100:.1f}%")

Hybrid paragraph + sliding window: 542 docs -> 14745 chunks
Average: 27.2 chunks per doc

Paragraphs analyzed: 14254
Paragraphs exceeding threshold (triggered sliding window): 151
Trigger rate: 1.1%


### Section-Based Chunking

Section chunking uses markdown structure (## headers) to find logical boundaries. Each section includes nested subsections until the next ## header.

In [17]:
def chunk_by_section(doc: dict[str, Any]) -> list[dict[str, Any]]:
    """Split document by ## markdown headers.

    Each chunk contains one ## section with all nested subsections (###, ####, etc.)
    until the next ## header. The header text is included in the chunk.

    Args:
        doc: Document dict with 'content', 'filename', and optional 'metadata' keys.

    Returns:
        List of chunk dicts with same structure as chunk_by_paragraph().
        If no ## headers found, returns single chunk with entire document.

    Raises:
        KeyError: If doc missing 'content' or 'filename' keys.

    Example:
        >>> doc = {'filename': 'test.md', 'metadata': {}, 'content': '## A\\nText\\n## B\\nMore'}
        >>> chunks = chunk_by_section(doc)
        >>> len(chunks)
        2
        >>> chunks[0]['content'].startswith('## A')
        True
    """
    # Find all ## headers and their positions
    pattern = r'^##\s+(.+)$'
    matches = list(re.finditer(pattern, doc['content'], re.MULTILINE))

    if not matches:
        # No sections found, return whole doc as single chunk
        return [{
            'filename': doc['filename'],
            'metadata': copy.deepcopy(doc.get('metadata', {})),
            'content': doc['content'],
            'chunk_id': f"{doc['filename']}-chunk-0",
            'chunk_index': 0,
            'total_chunks': 1,
            'chunk_method': 'section'
        }]

    chunks = []
    for i, match in enumerate(matches):
        start = match.start()
        # End is start of next section or end of document
        end = matches[i+1].start() if i+1 < len(matches) else len(doc['content'])

        section_text = doc['content'][start:end].strip()

        # Skip empty sections (header-only)
        if not section_text:
            continue

        chunk = {
            'filename': doc['filename'],
            'metadata': copy.deepcopy(doc.get('metadata', {})),
            'content': section_text,  # includes header
            'chunk_id': f"{doc['filename']}-chunk-{len(chunks)}",
            'chunk_index': len(chunks),
            'total_chunks': -1,  # Updated after loop
            'chunk_method': 'section'
        }
        chunks.append(chunk)

    # Update total_chunks for all
    total = len(chunks)
    for chunk in chunks:
        chunk['total_chunks'] = total

    return chunks

### Testing Paragraph and Section Chunking on OWASP Docs

Apply both strategies to the OWASP documents and compare chunk counts.

In [18]:
# Apply paragraph chunking to all OWASP docs
paragraph_chunks = []
for doc in docs:
    paragraph_chunks.extend(chunk_by_paragraph(doc))

print(f"Paragraph chunking: {len(docs)} docs -> {len(paragraph_chunks)} chunks")
print(f"Average: {len(paragraph_chunks) / len(docs):.1f} chunks per doc")

# Apply section chunking to all OWASP docs
section_chunks = []
for doc in docs:
    section_chunks.extend(chunk_by_section(doc))

print(f"\nSection chunking: {len(docs)} docs -> {len(section_chunks)} chunks")
print(f"Average: {len(section_chunks) / len(docs):.1f} chunks per doc")

Paragraph chunking: 542 docs -> 14254 chunks
Average: 26.3 chunks per doc

Section chunking: 542 docs -> 1023 chunks
Average: 1.9 chunks per doc


### Sample Chunks and Metadata Verification

Verify that chunks have correct structure and preserve metadata.

In [19]:
# Verify chunk structure
if paragraph_chunks:
    sample = paragraph_chunks[0]
    print("=== Paragraph Chunk Sample ===")
    print(f"chunk_id: {sample['chunk_id']}")
    print(f"chunk_index: {sample['chunk_index']} of {sample['total_chunks']}")
    print(f"chunk_method: {sample['chunk_method']}")
    print(f"filename: {sample['filename']}")
    print(f"metadata keys: {list(sample['metadata'].keys())}")
    print(f"content length: {len(sample['content'])} chars")

    # Assertions for correctness
    assert sample['chunk_method'] == 'paragraph', "chunk_method should be 'paragraph'"
    assert isinstance(sample['chunk_index'], int), "chunk_index should be int"
    assert sample['chunk_index'] < sample['total_chunks'], "chunk_index < total_chunks"
    print("Assertions passed!")
    print()

if section_chunks:
    sample = section_chunks[0]
    print("=== Section Chunk Sample ===")
    print(f"chunk_id: {sample['chunk_id']}")
    print(f"chunk_index: {sample['chunk_index']} of {sample['total_chunks']}")
    print(f"chunk_method: {sample['chunk_method']}")
    print(f"filename: {sample['filename']}")
    print(f"metadata keys: {list(sample['metadata'].keys())}")
    print(f"content length: {len(sample['content'])} chars")

    # Assertions for correctness
    assert sample['chunk_method'] == 'section', "chunk_method should be 'section'"
    assert isinstance(sample['chunk_index'], int), "chunk_index should be int"
    assert sample['chunk_index'] < sample['total_chunks'], "chunk_index < total_chunks"
    print("Assertions passed!")

=== Paragraph Chunk Sample ===
chunk_id: www-project-top-10-for-large-language-model-applications-main/.github/PULL_REQUEST_TEMPLATE.md-chunk-0
chunk_index: 0 of 11
chunk_method: paragraph
filename: www-project-top-10-for-large-language-model-applications-main/.github/PULL_REQUEST_TEMPLATE.md
metadata keys: []
content length: 20 chars
Assertions passed!

=== Section Chunk Sample ===
chunk_id: www-project-top-10-for-large-language-model-applications-main/.github/PULL_REQUEST_TEMPLATE.md-chunk-0
chunk_index: 0 of 1
chunk_method: section
filename: www-project-top-10-for-large-language-model-applications-main/.github/PULL_REQUEST_TEMPLATE.md
metadata keys: []
content length: 569 chars
Assertions passed!


### Verifying Nested Subsection Preservation

Per D-09, section chunks must include all nested subsections (###, ####) within each ## section. OWASP docs are heavily structured with nested headers - let's verify they're preserved.

In [20]:
# Verify nested subsections are preserved in section chunks (D-09)
nested_header_pattern = r'^###\s+.+$'

# Find section chunks that contain nested ### headers
chunks_with_nested = []
for chunk in section_chunks:
    nested_matches = re.findall(nested_header_pattern, chunk['content'], re.MULTILINE)
    if nested_matches:
        chunks_with_nested.append({
            'chunk_id': chunk['chunk_id'],
            'nested_headers': nested_matches[:3],  # Show first 3
            'nested_count': len(nested_matches)
        })

print(f"Section chunks containing nested ### headers: {len(chunks_with_nested)} of {len(section_chunks)}")
print()

# Show examples of preserved nested structure
if chunks_with_nested:
    print("Examples of nested subsection preservation:")
    for item in chunks_with_nested[:3]:
        print(f"\n  {item['chunk_id']}")
        print(f"    Nested headers ({item['nested_count']} total): {item['nested_headers']}")

# Assertion: If any doc has nested headers, at least some section chunks should preserve them
# (This confirms D-09 behavior - nested subsections stay with parent ## section)
docs_with_nested = sum(1 for doc in docs if re.search(r'^###\s+', doc['content'], re.MULTILINE))
if docs_with_nested > 0:
    assert len(chunks_with_nested) > 0, "D-09 violation: nested ### headers not found in any section chunk"
    print(f"\nD-09 VERIFIED: Nested subsections preserved in section chunks")
else:
    print("\nNote: No nested ### headers found in source docs (D-09 not applicable)")


Section chunks containing nested ### headers: 461 of 1023

Examples of nested subsection preservation:

  www-project-top-10-for-large-language-model-applications-main/2_0_vulns/LLM00_Preface.md-chunk-0
    Nested headers (2 total): ['### What’s New in the 2025 Top 10', '### Moving Forward']

  www-project-top-10-for-large-language-model-applications-main/2_0_vulns/LLM01_PromptInjection.md-chunk-0
    Nested headers (6 total): ['### Description', '### Types of Prompt Injection Vulnerabilities', '### Prevention and Mitigation Strategies']

  www-project-top-10-for-large-language-model-applications-main/2_0_vulns/LLM02_SensitiveInformationDisclosure.md-chunk-0
    Nested headers (4 total): ['### Description', '### Common Examples of Vulnerability', '### Prevention and Mitigation Strategies']

D-09 VERIFIED: Nested subsections preserved in section chunks


### Strategy Comparison Framework

Compare all three chunking strategies using token-aware metrics and size distribution analysis.

In [21]:
import statistics
import tiktoken
from typing import Any

def compare_chunking_strategies(strategies: dict[str, list[dict[str, Any]]]) -> str:
    """Compare chunking strategies across multiple metrics.

    Calculates basic counts, token statistics, and size distribution percentiles
    for each strategy, formatted as a markdown table.

    Args:
        strategies: Dict mapping strategy name to list of chunk dicts.
                   Each chunk must have 'content' and 'filename' keys.

    Returns:
        Formatted markdown table string with strategies as columns, metrics as rows.
        Metrics include: total_chunks, avg_per_doc, avg_tokens, min_tokens,
        max_tokens, token_char_ratio, p25_chars, p50_chars, p75_chars, p95_chars.

    Example:
        >>> strategies = {'a': [{'content': 'text', 'filename': 'f'}]}
        >>> table = compare_chunking_strategies(strategies)
        >>> '| Metric |' in table
        True
    """
    enc = tiktoken.get_encoding('cl100k_base')

    results = {}
    for name, chunks in strategies.items():
        if not chunks:
            results[name] = {metric: 0 for metric in [
                'total_chunks', 'avg_per_doc', 'avg_tokens', 'min_tokens',
                'max_tokens', 'token_char_ratio', 'p25_chars', 'p50_chars',
                'p75_chars', 'p95_chars'
            ]}
            continue

        char_counts = [len(c['content']) for c in chunks]
        token_counts = [len(enc.encode(c['content'])) for c in chunks]

        # Count documents (unique filenames)
        doc_count = len(set(c['filename'] for c in chunks))

        # Calculate percentiles with edge case handling
        if len(char_counts) > 1:
            quartiles = statistics.quantiles(char_counts, n=4)
            p25 = quartiles[0]
            p75 = quartiles[2]
            ventiles = statistics.quantiles(char_counts, n=20)
            p95 = ventiles[18] if len(ventiles) > 18 else char_counts[-1]
        else:
            p25 = p75 = p95 = char_counts[0] if char_counts else 0

        p50 = statistics.median(char_counts) if char_counts else 0

        results[name] = {
            'total_chunks': len(chunks),
            'avg_per_doc': len(chunks) / doc_count if doc_count > 0 else 0,
            'avg_tokens': statistics.mean(token_counts) if token_counts else 0,
            'min_tokens': min(token_counts) if token_counts else 0,
            'max_tokens': max(token_counts) if token_counts else 0,
            'token_char_ratio': statistics.mean(token_counts) / statistics.mean(char_counts) if char_counts else 0,
            'p25_chars': p25,
            'p50_chars': p50,
            'p75_chars': p75,
            'p95_chars': p95,
        }

    # Format as markdown table
    strategy_names = list(strategies.keys())
    metrics = ['total_chunks', 'avg_per_doc', 'avg_tokens', 'min_tokens',
               'max_tokens', 'token_char_ratio', 'p25_chars', 'p50_chars',
               'p75_chars', 'p95_chars']

    header = "| Metric | " + " | ".join(strategy_names) + " |"
    separator = "|--------|" + "|".join(["--------"] * len(strategy_names)) + "|"

    rows = []
    for metric in metrics:
        row_vals = []
        for name in strategy_names:
            val = results[name][metric]
            if isinstance(val, float):
                row_vals.append(f"{val:.2f}")
            else:
                row_vals.append(str(val))
        rows.append(f"| {metric} | " + " | ".join(row_vals) + " |")

    table = "\n".join([header, separator] + rows)
    return table

### Chunk Quality Inspection

Detect potential quality issues: boundary problems, unclosed code blocks, size outliers, and orphaned references.

In [22]:
def inspect_chunk_quality(
    chunks: list[dict[str, Any]],
    strategy_name: str
) -> list[dict[str, Any]]:
    """Detect quality issues in chunks.

    Scans chunks for boundary problems (incomplete sentences, unclosed code blocks),
    size outliers (>2x or <0.5x mean), and orphaned references.

    Args:
        chunks: List of chunk dicts, each with 'content' key.
        strategy_name: Name of chunking strategy (for context in output).

    Returns:
        List of flagged samples, each containing:
        - 'chunk': The original chunk dict
        - 'issues': List of detected issue descriptions
        Limited to 5 samples maximum.

    Example:
        >>> chunks = [{'content': '```python\\ncode', 'chunk_id': 'test'}]
        >>> flagged = inspect_chunk_quality(chunks, 'test')
        >>> 'unclosed code block' in flagged[0]['issues']
        True
    """
    if not chunks:
        return []

    flagged = []
    chunk_sizes = [len(c['content']) for c in chunks]
    mean_size = statistics.mean(chunk_sizes)

    for chunk in chunks:
        issues = []
        text = chunk['content']

        # Boundary problems
        if text and not text.rstrip().endswith(('.', '!', '?', '```', ')', ']', ':', '-')):
            if not text.strip().startswith(('##', '-', '*', '1.', '>')):
                issues.append('mid-word or incomplete sentence')

        # Broken code blocks
        if text.count('```') % 2 != 0:
            issues.append('unclosed code block')

        # Orphaned references
        if re.search(r'\b(above|below|previous|next|following)\b', text.lower()):
            issues.append('orphaned reference')

        # Size outliers
        size = len(text)
        if size > mean_size * 2:
            issues.append(f'too large ({size / mean_size:.1f}x mean)')
        elif size < mean_size * 0.5:
            issues.append(f'too small ({size / mean_size:.2f}x mean)')

        if issues:
            flagged.append({'chunk': chunk, 'issues': issues})

    return flagged[:5]

### OWASP Chunking Strategy Comparison

Compare all three strategies on the OWASP documentation corpus.

In [23]:
# Collect all chunk sets
strategies = {
    'sliding_window': sliding_window_chunks,
    'paragraph': paragraph_chunks,
    'section': section_chunks
}

comparison_table = compare_chunking_strategies(strategies)
print(comparison_table)

| Metric | sliding_window | paragraph | section |
|--------|--------|--------|--------|
| total_chunks | 3563 | 14254 | 1023 |
| avg_per_doc | 6.61 | 26.45 | 1.89 |
| avg_tokens | 553.38 | 75.01 | 1045.13 |
| min_tokens | 1 | 1 | 0 |
| max_tokens | 2330 | 43321 | 43324 |
| token_char_ratio | 0.33 | 0.33 | 0.33 |
| p25_chars | 1651.00 | 26.00 | 377.00 |
| p50_chars | 2000 | 63.00 | 1598 |
| p75_chars | 2000.00 | 259.00 | 4755.00 |
| p95_chars | 2000.00 | 828.25 | 9981.80 |


In [24]:
# Updated comparison with hybrid strategy
strategies = {
    'sliding_window': sliding_window_chunks,
    'paragraph': paragraph_chunks,
    'section': section_chunks,
    'hybrid': hybrid_chunks
}

comparison_table_with_hybrid = compare_chunking_strategies(strategies)
print(comparison_table_with_hybrid)

| Metric | sliding_window | paragraph | section | hybrid |
|--------|--------|--------|--------|--------|
| total_chunks | 3563 | 14254 | 1023 | 14745 |
| avg_per_doc | 6.61 | 26.45 | 1.89 | 27.36 |
| avg_tokens | 553.38 | 75.01 | 1045.13 | 81.49 |
| min_tokens | 1 | 1 | 0 | 1 |
| max_tokens | 2330 | 43321 | 43324 | 1958 |
| token_char_ratio | 0.33 | 0.33 | 0.33 | 0.33 |
| p25_chars | 1651.00 | 26.00 | 377.00 | 27.00 |
| p50_chars | 2000 | 63.00 | 1598 | 73 |
| p75_chars | 2000.00 | 259.00 | 4755.00 | 282.00 |
| p95_chars | 2000.00 | 828.25 | 9981.80 | 1154.00 |


### OWASP Quality Inspection Results

In [25]:
for name, chunks in strategies.items():
    print(f"\n=== {name.upper()} Quality Inspection ===")
    flagged = inspect_chunk_quality(chunks, name)

    if not flagged:
        print("No quality issues detected in sample.")
    else:
        print(f"Found {len(flagged)} potentially problematic chunks:")
        for i, item in enumerate(flagged):
            chunk = item['chunk']
            issues = item['issues']
            print(f"\n  [{i+1}] {chunk['chunk_id']}")
            print(f"      Issues: {', '.join(issues)}")
            print(f"      Preview: {chunk['content'][:100]}...")


=== SLIDING_WINDOW Quality Inspection ===
Found 5 potentially problematic chunks:

  [1] www-project-top-10-for-large-language-model-applications-main/.github/PULL_REQUEST_TEMPLATE.md-chunk-0
      Issues: mid-word or incomplete sentence, too small (0.34x mean)
      Preview: # [Title of Your PR]

**Key Changes:**

- [ ] List major changes and core updates
- [ ] Keep each li...

  [2] www-project-top-10-for-large-language-model-applications-main/2_0_vulns/LLM00_Preface.md-chunk-1
      Issues: mid-word or incomplete sentence
      Preview: dback. Each voice was critical to making this new release as thorough and practical as possible.

##...

  [3] www-project-top-10-for-large-language-model-applications-main/2_0_vulns/LLM00_Preface.md-chunk-2
      Issues: mid-word or incomplete sentence
      Preview: remains secret.

**Excessive Agency** has been expanded, given the increased use of agentic architec...

  [4] www-project-top-10-for-large-language-model-applications-main/2_0_vulns/L

In [26]:
# Inspect hybrid chunk quality
print("=== HYBRID PARAGRAPH + SLIDING WINDOW Quality Inspection ===")
flagged = inspect_chunk_quality(hybrid_chunks, 'hybrid')

if not flagged:
    print("No quality issues detected in sample.")
else:
    print(f"Found {len(flagged)} potentially problematic chunks:")
    for i, item in enumerate(flagged):
        chunk = item['chunk']
        issues = item['issues']
        print(f"\n  [{i+1}] {chunk['chunk_id'][:60]}...")
        print(f"      Issues: {', '.join(issues)}")
        print(f"      Preview: {chunk['content'][:100]}...")

=== HYBRID PARAGRAPH + SLIDING WINDOW Quality Inspection ===
Found 5 potentially problematic chunks:

  [1] www-project-top-10-for-large-language-model-applications-mai...
      Issues: too small (0.08x mean)
      Preview: # [Title of Your PR]...

  [2] www-project-top-10-for-large-language-model-applications-mai...
      Issues: too small (0.06x mean)
      Preview: **Key Changes:**...

  [3] www-project-top-10-for-large-language-model-applications-mai...
      Issues: too small (0.48x mean)
      Preview: - [ ] List major changes and core updates
- [ ] Keep each line under 80 characters
- [ ] Focus on th...

  [4] www-project-top-10-for-large-language-model-applications-mai...
      Issues: too small (0.04x mean)
      Preview: **Added:**...

  [5] www-project-top-10-for-large-language-model-applications-mai...
      Issues: too small (0.35x mean)
      Preview: - [ ] New features/functionality
- [ ] New files/configurations
- [ ] New dependencies...


### OWASP Chunking Analysis

**OWASP documentation characteristics:**
- 542 markdown documents with NO frontmatter (plain markdown)
- Heavy ## structure: security topics (LLM01, LLM02, etc.) with nested subsections
- Many code examples, configuration snippets, and cross-references
- Nested subsections (###) preserved in section chunking (461 of 1023 chunks contain nested headers)

**Quantitative Comparison**

From the comparison tables above:

| Metric | Sliding Window | Paragraph | Section | Hybrid |
|--------|----------------|-----------|---------|--------|
| Total chunks | 3563 | 14254 | 1023 | See table |
| Avg per doc | 6.6 | 26.3 | 1.9 | See table |
| Avg tokens | 553 | 75 | 1045 | See table |
| Max tokens | 2330 | 43321 | 43324 | See table |
| p50 chars | 2000 | 63 | 1598 | See table |
| p95 chars | 2000 | 828 | 9982 | See table |

**Interpretation:**
- **Paragraph** produced most chunks (14254) due to many short bullet points and code snippets in OWASP structure
- **Paragraph** has extreme max tokens (43321) from occasional very large paragraphs
- **Section** has fewest chunks (1023) and highest avg tokens (1045) - logical topic boundaries create larger, coherent chunks
- **Hybrid** reduces paragraph's size outliers by applying sliding window to oversized paragraphs
- **Hybrid trigger rate:** See cell output above — shows percentage of paragraphs that exceeded the 2000-char threshold and triggered sliding window subdivision

**Strategy Recommendation for OWASP**

Primary recommendation: **Section chunking**

Rationale:
- OWASP docs have clear ## structure (LLM01: Prompt Injection, LLM02: Sensitive Information Disclosure, etc.)
- Each ## section represents one security vulnerability - preserving this boundary is critical for retrieval accuracy
- Nested subsections (###) stay with parent section (verified: 461 of 1023 chunks contain nested headers)
- Moderate chunk sizes: median 1598 chars, avg 1045 tokens - fits embedding models well
- Quality issues minimal compared to paragraph (no extreme size outliers due to semantic completeness)

When to use alternatives for OWASP:
- **Sliding window:** When fixed-size embeddings are required by embedding model constraints
- **Paragraph:** Not recommended for OWASP (produces 14K chunks with high size variance)
- **Hybrid paragraph + sliding window:** Addresses paragraph's size outliers (see trigger rate above), but still produces many small chunks from bullet points
- **LLM:** Cost not justified - OWASP structure is clear enough that section chunking captures semantic boundaries for free

**Decision Framework**

Generalized decision tree for choosing chunking strategy:

1. **If structured documentation with ## markdown headers → Section chunking**
   - Examples: API docs, technical guides, security documentation (OWASP)
   - Preserves topic coherence, includes headers for context retrieval
   - Best when document structure aligns with semantic topics

2. **If natural paragraphs but occasional very large sections → Hybrid (paragraph + sliding window)**
   - Examples: Mixed content (prose + code), long-form articles with code blocks
   - Respects paragraph boundaries where possible, prevents size outliers
   - Useful when paragraph chunking produces chunks > 5000 chars

3. **If narrative content without clear structure → Paragraph chunking**
   - Examples: Blog posts, articles, prose narratives
   - Semantic completeness at natural boundaries
   - Not ideal for technical docs with bullet points (produces many tiny chunks)

4. **If consistent chunk sizes required → Sliding window**
   - Examples: Embedding models with strict token limits, batch processing
   - Predictable sizes, simple implementation
   - Structure-blind - will split mid-sentence and mid-code-block

5. **If quality is critical and budget allows → LLM chunking**
   - Examples: High-value content requiring semantic understanding
   - Costs ~$0.001-0.01 per document depending on size
   - Consider Groq (free tier) for testing

**OWASP-Specific Conclusion:**

Section chunking is optimal for OWASP Top 10 documentation because the heavy ## structure aligns perfectly with security vulnerability topics. Retrieval benefits from topic coherence (a user query about "prompt injection" should retrieve the entire LLM01 section). Chunk sizes are moderate and consistent. The hybrid approach provides minimal benefit because paragraph chunking's size outliers are rare enough (see trigger rate from cell above) that the added complexity isn't justified.

### LLM-Based Chunking

Use LLM intelligence to identify semantic boundaries. The LLM analyzes document content and returns character positions where natural topic breaks occur.

**Implementation notes:**
- Supports both Groq (default, free tier) and OpenAI providers
- Returns JSON boundaries for exact text preservation
- Includes retry logic with exponential backoff
- Falls back to paragraph chunking after 3 failures

In [27]:
# LLM chunking imports
import os
import time
import json
from openai import OpenAI
from groq import Groq

In [28]:
def split_at_boundaries(
    doc: dict[str, Any],
    boundaries: list[int],
    provider: str
) -> list[dict[str, Any]]:
    """Split document at given character positions.

    Creates chunk dicts from document content using the specified boundary
    positions. Preserves all source metadata via deep copy.

    Args:
        doc: Source document dict with 'content', 'filename', 'metadata' keys.
        boundaries: List of character positions where splits should occur.
            Positions should be within document content length.
        provider: Provider identifier ('groq' or 'openai') for chunk_method.

    Returns:
        List of chunk dicts, each containing:
        - filename: Inherited from source document
        - metadata: Deep copy of source metadata
        - content: Chunk text (trimmed whitespace)
        - chunk_id: Format '{filename}-chunk-{index}'
        - chunk_index: Position in sequence (0-indexed)
        - total_chunks: Total chunks created from document
        - chunk_method: Format 'llm_{provider}'

    Example:
        >>> doc = {'filename': 'test.md', 'metadata': {}, 'content': 'ABC DEF GHI'}
        >>> chunks = split_at_boundaries(doc, [4, 8], 'groq')
        >>> [c['content'] for c in chunks]
        ['ABC', 'DEF', 'GHI']
    """
    content = doc['content']
    chunks: list[dict[str, Any]] = []

    # Add start (0) and end (len) to boundaries
    positions = [0] + sorted(boundaries) + [len(content)]

    for i in range(len(positions) - 1):
        start = positions[i]
        end = positions[i + 1]
        chunk_text = content[start:end].strip()

        if chunk_text:  # Skip empty
            chunk = {
                'filename': doc['filename'],
                'metadata': copy.deepcopy(doc.get('metadata', {})),
                'content': chunk_text,
                'chunk_id': f"{doc['filename']}-chunk-{i}",
                'chunk_index': i,
                'total_chunks': -1,  # Updated after loop
                'chunk_method': f'llm_{provider}'
            }
            chunks.append(chunk)

    # Update total_chunks for all
    total = len(chunks)
    for chunk in chunks:
        chunk['total_chunks'] = total

    return chunks

In [29]:
def chunk_by_llm(
    doc: dict[str, Any],
    provider: str = 'groq'
) -> list[dict[str, Any]]:
    """Split document using LLM semantic boundary detection.

    Uses OpenAI or Groq API to identify natural topic boundaries in text.
    The LLM returns character positions for splits, preserving original
    text exactly (no rewriting). Includes retry logic with exponential
    backoff for transient errors.

    Args:
        doc: Document dict with required keys:
            - 'content' (str): Document text to analyze
            - 'filename' (str): Source document identifier
            - 'metadata' (dict, optional): Frontmatter metadata to preserve
        provider: API provider to use. Options:
            - 'groq': Uses llama-3.1-8b-instant (default, free tier available)
            - 'openai': Uses gpt-4o-mini

    Returns:
        List of chunk dicts with same structure as other chunking functions.
        Each chunk has chunk_method='llm_groq' or 'llm_openai'.

    Raises:
        ValueError: If API key not found in environment variables.
            - GROQ_API_KEY for provider='groq'
            - OPENAI_API_KEY for provider='openai'

    Side Effects:
        - Prints per-document cost information (tokens, dollars)
        - Makes HTTP requests to LLM API

    Example:
        >>> import os
        >>> os.environ['GROQ_API_KEY'] = 'gsk_...'  # Set API key
        >>> doc = {'filename': 'test.md', 'metadata': {}, 'content': 'Long text...'}
        >>> chunks = chunk_by_llm(doc, provider='groq')
        >>> chunks[0]['chunk_method']
        'llm_groq'

    Notes:
        - Groq costs: ~$0.05/1M input tokens, ~$0.08/1M output tokens
        - OpenAI costs: ~$0.15/1M input tokens, ~$0.60/1M output tokens
        - Falls back to paragraph chunking after 3 API failures
    """
    # Step 1: Check API keys (per D-02, D-03)
    key_name = 'GROQ_API_KEY' if provider == 'groq' else 'OPENAI_API_KEY'
    api_key = os.getenv(key_name)
    if not api_key:
        raise ValueError(f"{key_name} not found - set via: op run --env-file=...")

    # Step 2: Initialize client and select model (per D-04, D-05)
    if provider == 'groq':
        client = Groq(api_key=api_key)
        model = 'llama-3.1-8b-instant'
        input_rate = 0.05   # $/1M tokens
        output_rate = 0.08  # $/1M tokens
    else:
        client = OpenAI(api_key=api_key)
        model = 'gpt-4o-mini'
        input_rate = 0.15   # $/1M tokens
        output_rate = 0.60  # $/1M tokens

    # Step 3: Build prompt (per D-06 through D-10)
    system_msg = "You are a document chunking expert."
    user_msg = f"""Identify semantic boundaries in this document.

Rules:
- Split at topic changes and after complete thoughts
- Avoid mid-sentence breaks
- Keep code blocks intact
- Aim for chunks around 500-1000 tokens, but prioritize semantic completeness

Return character positions as JSON: {{"boundaries": [120, 450, 890]}}

Document:
{doc['content']}"""

    # Step 4: Call API with retry logic (per D-14, D-15, D-16)
    for attempt in range(3):
        try:
            response = client.chat.completions.create(
                model=model,
                messages=[
                    {"role": "system", "content": system_msg},
                    {"role": "user", "content": user_msg}
                ],
                response_format={"type": "json_object"},
                temperature=0.3
            )

            # Parse JSON boundaries (per D-07)
            result = json.loads(response.choices[0].message.content)
            boundaries = result.get('boundaries', [])

            # Extract usage for cost tracking (per D-11, D-12)
            usage = response.usage
            total_tokens = usage.total_tokens
            cost = (usage.prompt_tokens * input_rate + usage.completion_tokens * output_rate) / 1_000_000

            print(f"Document: {doc['filename'][:50]}... - Tokens: {total_tokens}, Cost: ${cost:.6f}")

            # Split document at boundaries
            chunks = split_at_boundaries(doc, boundaries, provider)
            return chunks

        except Exception as e:
            if '429' in str(e):  # Rate limit (per D-15)
                retry_after = getattr(e, 'retry_after', 2 ** attempt)
                print(f"Rate limit, waiting {retry_after}s...")
                time.sleep(retry_after)
            elif 'json' in str(e).lower():  # JSON parse error (per D-16)
                print(f"JSON parse error, retrying with stricter prompt...")
                user_msg += "\n\nONLY output valid JSON."
            else:
                print(f"Attempt {attempt + 1} failed: {e}")
                time.sleep(2 ** attempt)

    # Fallback to paragraph chunking after 3 failures (per D-16)
    print(f"LLM chunking failed, falling back to paragraph chunking")
    return chunk_by_paragraph(doc)

### Running LLM Chunking on OWASP Docs

**Setup:** Run notebook with API key via OnePassword injection:
```bash
op run --env-file=.env jupyter notebook
```

**Note:** OWASP has 542 documents - processing all will cost approximately:
- Groq: ~$0.10-0.50 (free tier has 500k tokens/day limit)
- OpenAI: ~$0.30-1.50

Consider testing on a subset first.

In [30]:
# Test on a single OWASP document first
if os.getenv('GROQ_API_KEY') or os.getenv('OPENAI_API_KEY'):
    provider = 'groq' if os.getenv('GROQ_API_KEY') else 'openai'
    sample_doc = docs[0]
    sample_chunks = chunk_by_llm(sample_doc, provider=provider)
    print(f"\nSample document chunked into {len(sample_chunks)} chunks")
    print(f"Chunk method: {sample_chunks[0]['chunk_method'] if sample_chunks else 'N/A'}")

    # Verify structure
    if sample_chunks:
        chunk = sample_chunks[0]
        assert 'filename' in chunk, "Missing filename"
        assert 'metadata' in chunk, "Missing metadata"
        assert 'content' in chunk, "Missing content"
        assert chunk['chunk_method'].startswith('llm_'), "Invalid chunk_method"
        print("Structure validation passed!")
else:
    print("No API key found. Set GROQ_API_KEY or OPENAI_API_KEY to run LLM chunking.")

No API key found. Set GROQ_API_KEY or OPENAI_API_KEY to run LLM chunking.


In [31]:
# Chunk a subset of OWASP documents (10 docs to manage costs)
llm_chunks: list[dict[str, Any]] = []
if os.getenv('GROQ_API_KEY') or os.getenv('OPENAI_API_KEY'):
    provider = 'groq' if os.getenv('GROQ_API_KEY') else 'openai'
    subset = docs[:10]  # First 10 documents
    print(f"Using {provider} provider")
    print(f"Processing {len(subset)} of {len(docs)} documents...\n")

    for doc in subset:
        chunks = chunk_by_llm(doc, provider=provider)
        llm_chunks.extend(chunks)

    print(f"\n{'='*60}")
    print(f"Total: {len(llm_chunks)} chunks from {len(subset)} documents")
else:
    print("Skipping LLM chunking - no API key available.")

Skipping LLM chunking - no API key available.


In [32]:
# Compare chunking strategies
# Note: For fair comparison, we need to compare same documents
# Using first 10 docs for all strategies
if llm_chunks:
    # Re-chunk first 10 docs with other strategies for fair comparison
    subset = docs[:10]

    sw_subset = []
    para_subset = []
    sect_subset = []
    for doc in subset:
        sw_subset.extend(chunk_sliding_window(doc))
        para_subset.extend(chunk_by_paragraph(doc))
        sect_subset.extend(chunk_by_section(doc))

    strategies_with_llm = {
        'sliding_window': sw_subset,
        'paragraph': para_subset,
        'section': sect_subset,
        'llm': llm_chunks
    }
    comparison_with_llm = compare_chunking_strategies(strategies_with_llm)
    print("Comparison on first 10 OWASP documents:")
    print(comparison_with_llm)
else:
    print("LLM chunks not available - comparison skipped.")

LLM chunks not available - comparison skipped.


In [33]:
# Inspect LLM chunk quality for OWASP docs
if llm_chunks:
    print("=== LLM Chunking Quality Inspection (OWASP) ===")
    llm_flagged = inspect_chunk_quality(llm_chunks, 'llm')

    if not llm_flagged:
        print("No quality issues detected in sample.")
    else:
        print(f"Found {len(llm_flagged)} potentially problematic chunks:")
        for i, item in enumerate(llm_flagged):
            chunk = item['chunk']
            issues = item['issues']
            print(f"\n  [{i+1}] {chunk['chunk_id'][:60]}...")
            print(f"      Issues: {', '.join(issues)}")
            print(f"      Preview: {chunk['content'][:100]}...")
else:
    print("LLM chunks not available - quality inspection skipped.")

LLM chunks not available - quality inspection skipped.


### LLM Chunking Observations (OWASP)

**OWASP-Specific Findings:**
- OWASP security documents have clear ## structure (vulnerability categories)
- LLM may not significantly improve on section chunking for well-structured docs
- Cost is substantial for 542 documents - subset testing recommended

**Comparison Notes:**
- Section chunking: Free, respects security topic boundaries
- LLM chunking: Costly, may preserve cross-references better
- Recommendation: For OWASP structure, section chunking likely sufficient

**When LLM chunking might help:**
- Documents without clear headers (narrative security reports)
- Mixed content requiring semantic understanding (code + explanation)
- When retrieval quality is worth the API cost

___

## Day 3: Search

Apply Day 3 search methods to OWASP LLM Top 10 corpus. Uses multi-granularity chunking: section chunks (1,023) for text search, paragraph chunks (14,254) for vector search.

In [ ]:
# Day 3 imports
from typing import Any
import minsearch
from minsearch import VectorSearch
from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity
import numpy as np
from pathlib import Path

print("Day 3 imports loaded")

In [ ]:
def prepare_for_indexing(chunks: list[dict[str, Any]]) -> list[dict[str, Any]]:
    """Add title field to chunks for consistent indexing.

    Args:
        chunks: List of chunk dicts with metadata, content, filename keys.

    Returns:
        List of chunks with 'title' field added (from metadata or filename).

    Example:
        >>> prepared = prepare_for_indexing(section_chunks)
        >>> prepared[0]['title']  # Returns title from metadata or filename
    """
    prepared = []
    for chunk in chunks:
        title = chunk['metadata'].get('title', chunk['filename'])

        prepared_chunk = {
            'content': chunk['content'],
            'title': title,
            'filename': chunk['filename'],
            'chunk_id': chunk['chunk_id'],
            'metadata': chunk['metadata']
        }
        prepared.append(prepared_chunk)

    return prepared

print("prepare_for_indexing() defined")

In [ ]:
def text_search(index: minsearch.Index, query: str, top_k: int = 5) -> list[dict[str, Any]]:
    """Search index using text/lexical matching with TF-IDF scoring.

    Args:
        index: minsearch.Index instance with fitted documents.
        query: Search query string.
        top_k: Number of results to return.

    Returns:
        List of dicts with content, title, filename, chunk_id, metadata.
        Sorted by TF-IDF relevance score (highest first).

    Example:
        >>> results = text_search(owasp_index, "LLM01", top_k=3)
        >>> results[0]['title']  # Top result
    """
    results = index.search(
        query=query,
        boost_dict={"title": 2.0, "content": 1.0},
        num_results=top_k
    )
    return results

print("text_search() defined")

### Text Search on OWASP Section Chunks

Index 1,023 section chunks for text/lexical search. Section chunks preserve complete vulnerability topics (LLM01, LLM02, etc.) with all subsections - optimal for TF-IDF which needs sufficient context for term statistics.

In [ ]:
# Prepare section chunks for indexing (adds title field)
owasp_prepared = prepare_for_indexing(section_chunks)
print(f"Prepared {len(owasp_prepared)} section chunks for indexing")

# Create text search index
owasp_index = minsearch.Index(
    text_fields=["content", "title"],
    keyword_fields=["filename", "chunk_id"]
)

# Fit index with prepared chunks
owasp_index.fit(owasp_prepared)
print(f"Indexed {len(owasp_prepared)} OWASP section chunks")

In [ ]:
# Test text search with LLM01 (exact vulnerability code)
results = text_search(owasp_index, "LLM01", top_k=3)

print("Text Search: 'LLM01' query")
print("=" * 60)
for i, result in enumerate(results, 1):
    print(f"{i}. {result['filename']}")
    print(f"   Preview: {result['content'][:150]}...")
    print()

# Verify LLM01 appears in top result
assert any("LLM01" in r['content'] for r in results), "LLM01 should appear in results"
print("Text search verified: LLM01 found in results")

### Vector Search on OWASP Paragraph Chunks

Index 14,254 paragraph chunks for semantic search. Paragraph chunks (~75 tokens avg) are optimal for embedding models trained on sentence pairs - providing precise semantic matching.

In [ ]:
def get_or_generate_embeddings(
    chunks: list[dict[str, Any]],
    cache_path: Path,
    model: SentenceTransformer,
    batch_size: int = 32
) -> np.ndarray:
    """Load embeddings from cache or generate if not present.

    Args:
        chunks: List of chunk dicts with 'content' key.
        cache_path: Path to .npy cache file.
        model: SentenceTransformer model instance.
        batch_size: Chunks per batch for generation.

    Returns:
        numpy array of embeddings with shape (N, 384).

    Example:
        >>> embeddings = get_or_generate_embeddings(chunks, cache_path, model)
        >>> embeddings.shape  # (14254, 384)
    """
    if cache_path.exists():
        print(f"Loading cached embeddings from {cache_path}...")
        embeddings = np.load(cache_path)
        print(f"Loaded {embeddings.shape[0]} embeddings ({embeddings.shape[1]}-dim)")
        return embeddings

    # Generate new embeddings
    texts = [chunk['content'] for chunk in chunks]
    print(f"Generating embeddings for {len(texts)} chunks (batch_size={batch_size})...")
    embeddings = model.encode(
        texts,
        batch_size=batch_size,
        show_progress_bar=True
    )

    # Cache to disk
    np.save(cache_path, embeddings)
    print(f"Cached embeddings to {cache_path}")
    print(f"Shape: {embeddings.shape}")

    return embeddings

print("get_or_generate_embeddings() defined")

In [ ]:
def vector_search(
    index: VectorSearch,
    query: str,
    model: SentenceTransformer,
    top_k: int = 5
) -> list[dict[str, Any]]:
    """Search using VectorSearch with cosine similarity.

    Args:
        index: VectorSearch index with pre-computed embeddings.
        query: Search query string.
        model: SentenceTransformer model for query encoding.
        top_k: Number of results to return.

    Returns:
        List of dicts with content, title, filename, chunk_id, metadata, score.
        Score is cosine similarity (0.0-1.0, higher is more similar).

    Example:
        >>> results = vector_search(owasp_vector_index, "prompt injection", model)
        >>> results[0]['score']  # Cosine similarity score
    """
    query_embedding = model.encode([query])[0]
    results = index.search(query_embedding, num_results=top_k, output_ids=True)

    query_2d = query_embedding.reshape(1, -1)
    for result in results:
        result_id = result.pop('_id')
        result_embedding = index.vectors[result_id]
        score = cosine_similarity(query_2d, result_embedding.reshape(1, -1))[0][0]
        result['score'] = float(score)

    return results

print("vector_search() defined")

In [ ]:
# Load sentence-transformers model (all-MiniLM-L6-v2: 384-dim, 22MB)
print("Loading sentence-transformers model...")
model = SentenceTransformer('all-MiniLM-L6-v2')
print(f"Model loaded: {model.get_sentence_embedding_dimension()}-dimensional embeddings")

# Prepare paragraph chunks for indexing
owasp_para_prepared = prepare_for_indexing(paragraph_chunks)
print(f"Prepared {len(owasp_para_prepared)} paragraph chunks for vector search")

In [ ]:
# Set up embeddings cache
embeddings_dir = Path("data/embeddings")
embeddings_dir.mkdir(parents=True, exist_ok=True)
owasp_para_cache = embeddings_dir / "owasp_paragraphs.npy"

# Generate or load cached embeddings
# First run: ~3-4 minutes for 14,254 chunks
# Subsequent runs: <1 second from cache
owasp_para_embeddings = get_or_generate_embeddings(
    owasp_para_prepared,
    owasp_para_cache,
    model,
    batch_size=32
)
print(f"Embeddings shape: {owasp_para_embeddings.shape}")

In [ ]:
# Create VectorSearch index with paragraph embeddings
owasp_vector_index = VectorSearch(
    keyword_fields=["filename", "chunk_id"]
)
owasp_vector_index.fit(owasp_para_embeddings, owasp_para_prepared)
print(f"Created vector index with {len(owasp_para_prepared)} paragraph chunks")

### Hybrid Search with Multi-Granularity RRF

Combines section-level text search with paragraph-level vector search using Reciprocal Rank Fusion (k=60). Maps paragraph results back to parent sections for unified retrieval.

In [ ]:
def build_para_to_section_map(
    paragraph_chunks: list[dict[str, Any]],
    section_chunks: list[dict[str, Any]]
) -> dict[str, str]:
    """Build mapping from paragraph chunk_id to parent section chunk_id.

    Args:
        paragraph_chunks: List of paragraph chunk dicts (14,254 for OWASP).
        section_chunks: List of section chunk dicts (1,023 for OWASP).

    Returns:
        Dict mapping paragraph chunk_id to section chunk_id.
        Paragraphs without parent map to themselves (fallback).

    Example:
        >>> mapping = build_para_to_section_map(paragraph_chunks, section_chunks)
        >>> mapping['LLM01...-chunk-5']  # Returns parent section chunk_id
    """
    # Group sections by filename for faster lookup
    sections_by_file: dict[str, list[dict[str, Any]]] = {}
    for section in section_chunks:
        filename = section['filename']
        if filename not in sections_by_file:
            sections_by_file[filename] = []
        sections_by_file[filename].append(section)

    # Map each paragraph to parent section
    mapping: dict[str, str] = {}
    for para in paragraph_chunks:
        para_id = para['chunk_id']
        filename = para['filename']

        # Default: map to self if no parent found
        parent_section_id = para_id

        if filename in sections_by_file:
            for section in sections_by_file[filename]:
                # Check if paragraph content appears in section content
                if para['content'] in section['content']:
                    parent_section_id = section['chunk_id']
                    break

        mapping[para_id] = parent_section_id

    return mapping

print("build_para_to_section_map() defined")

In [ ]:
# Build paragraph -> section mapping
para_to_section_map = build_para_to_section_map(paragraph_chunks, section_chunks)
print(f"Built mapping for {len(para_to_section_map)} paragraphs")

# Create section chunks dict for result assembly
section_chunks_dict = {chunk['chunk_id']: chunk for chunk in section_chunks}
print(f"Created section lookup dict with {len(section_chunks_dict)} entries")

In [ ]:
def hybrid_search(
    text_index: minsearch.Index,
    vector_index: VectorSearch,
    query: str,
    model: SentenceTransformer,
    para_to_section_map: dict[str, str],
    section_chunks_dict: dict[str, dict[str, Any]],
    top_k: int = 5
) -> list[dict[str, Any]]:
    """Hybrid search with multi-granularity RRF fusion.

    Combines section-level text search with paragraph-level vector search.
    Deduplicates by mapping paragraph results to parent section chunks.

    Args:
        text_index: Text search index (section chunks).
        vector_index: Vector search index (paragraph chunks).
        query: Search query string.
        model: SentenceTransformer for query encoding.
        para_to_section_map: Maps paragraph chunk_id to section chunk_id.
        section_chunks_dict: Section chunks keyed by chunk_id.
        top_k: Number of results to return.

    Returns:
        List of section chunk dicts with normalized RRF scores (0.0-1.0).

    Example:
        >>> results = hybrid_search(owasp_index, owasp_vector_index, "LLM01 prevention", model, para_to_section_map, section_chunks_dict)
        >>> results[0]['score']  # Normalized RRF score
    """
    # 1. Get results from both methods
    text_results = text_search(text_index, query, top_k=top_k)
    vector_results = vector_search(vector_index, query, model, top_k=top_k)

    # 2. Compute RRF scores at section level
    k = 60
    rrf_scores: dict[str, float] = {}

    # Text contributes directly (already section-level)
    for rank, result in enumerate(text_results, start=1):
        section_id = result['chunk_id']
        rrf_scores[section_id] = rrf_scores.get(section_id, 0.0) + 1.0 / (k + rank)

    # Vector contributes via paragraph->section mapping
    for rank, result in enumerate(vector_results, start=1):
        para_id = result['chunk_id']
        section_id = para_to_section_map.get(para_id, para_id)
        rrf_scores[section_id] = rrf_scores.get(section_id, 0.0) + 1.0 / (k + rank)

    # 3. Assemble results from section_chunks_dict
    results = []
    for section_id, score in rrf_scores.items():
        if section_id in section_chunks_dict:
            section_chunk = section_chunks_dict[section_id].copy()
            section_chunk['score'] = score
            results.append(section_chunk)

    # 4. Normalize and sort
    if results:
        max_score = max(r['score'] for r in results)
        for r in results:
            r['score'] = r['score'] / max_score

    results.sort(key=lambda x: x['score'], reverse=True)
    return results[:top_k]

print("hybrid_search() defined")

In [ ]:
# Test hybrid search
test_results = hybrid_search(
    owasp_index,
    owasp_vector_index,
    "LLM01",
    model,
    para_to_section_map,
    section_chunks_dict,
    top_k=3
)

print("Hybrid Search Test: 'LLM01'")
print("=" * 60)
for i, result in enumerate(test_results, 1):
    print(f"{i}. [RRF: {result['score']:.3f}] {result['filename']}")
    print(f"   Preview: {result['content'][:100]}...")
    print()

print("Hybrid search verified")

### OWASP Search Experiments

Testing all three search methods on security-specific queries. Each query type demonstrates different search method strengths per D-11 through D-17.

In [ ]:
# Query 1: "LLM01" - Exact vulnerability code
# Expected: Text search excels (TF-IDF exact match)
print("=" * 80)
print("Query 1: 'LLM01' (Exact vulnerability code)")
print("Expected: Text search strength - direct keyword match")
print("=" * 80)

# Run all three methods
text_r1 = text_search(owasp_index, "LLM01", top_k=3)
vector_r1 = vector_search(owasp_vector_index, "LLM01", model, top_k=3)
hybrid_r1 = hybrid_search(owasp_index, owasp_vector_index, "LLM01", model, para_to_section_map, section_chunks_dict, top_k=3)

print("\n--- Text Search ---")
for i, r in enumerate(text_r1, 1):
    print(f"{i}. {r['filename'].split('/')[-1]}: {r['content'][:80]}...")

print("\n--- Vector Search ---")
for i, r in enumerate(vector_r1, 1):
    print(f"{i}. [Score: {r['score']:.3f}] {r['content'][:80]}...")

print("\n--- Hybrid Search ---")
for i, r in enumerate(hybrid_r1, 1):
    print(f"{i}. [RRF: {r['score']:.3f}] {r['filename'].split('/')[-1]}")

print("\nObservation: Text search finds LLM01 section immediately via exact term match.")

In [ ]:
# Query 2: "prompt injection" - Multi-word technical term
# Expected: Text search strength - phrase matching with field boosting
print("=" * 80)
print("Query 2: 'prompt injection' (Multi-word technical term)")
print("Expected: Text search strength - phrase matching")
print("=" * 80)

text_r2 = text_search(owasp_index, "prompt injection", top_k=3)
vector_r2 = vector_search(owasp_vector_index, "prompt injection", model, top_k=3)
hybrid_r2 = hybrid_search(owasp_index, owasp_vector_index, "prompt injection", model, para_to_section_map, section_chunks_dict, top_k=3)

print("\n--- Text Search ---")
for i, r in enumerate(text_r2, 1):
    has_term = "prompt injection" in r['content'].lower()
    print(f"{i}. Contains term: {has_term} | {r['content'][:60]}...")

print("\n--- Vector Search ---")
for i, r in enumerate(vector_r2, 1):
    print(f"{i}. [Score: {r['score']:.3f}] {r['content'][:60]}...")

print("\n--- Hybrid Search ---")
for i, r in enumerate(hybrid_r2, 1):
    print(f"{i}. [RRF: {r['score']:.3f}] {r['filename'].split('/')[-1]}")

print("\nObservation: Text search finds exact phrase 'prompt injection' with high precision.")

In [ ]:
# Query 3: "how to secure AI models" - Conceptual question
# Expected: Text search limitation, vector search strength
print("=" * 80)
print("Query 3: 'how to secure AI models' (Conceptual question)")
print("Expected: Vector search strength - semantic understanding")
print("=" * 80)

text_r3 = text_search(owasp_index, "how to secure AI models", top_k=3)
vector_r3 = vector_search(owasp_vector_index, "how to secure AI models", model, top_k=3)
hybrid_r3 = hybrid_search(owasp_index, owasp_vector_index, "how to secure AI models", model, para_to_section_map, section_chunks_dict, top_k=3)

print("\n--- Text Search ---")
for i, r in enumerate(text_r3, 1):
    print(f"{i}. {r['content'][:80]}...")

print("\n--- Vector Search ---")
for i, r in enumerate(vector_r3, 1):
    print(f"{i}. [Score: {r['score']:.3f}] {r['content'][:80]}...")

print("\n--- Hybrid Search ---")
for i, r in enumerate(hybrid_r3, 1):
    print(f"{i}. [RRF: {r['score']:.3f}] {r['filename'].split('/')[-1]}")

print("\nObservation: Vector search finds security/mitigation content via semantic matching.")

In [ ]:
# Query 4: "preventing malicious inputs" - Paraphrase
# Expected: Vector search strength - finds related content without exact keywords
print("=" * 80)
print("Query 4: 'preventing malicious inputs' (Paraphrase)")
print("Expected: Vector search strength - semantic paraphrase matching")
print("=" * 80)

text_r4 = text_search(owasp_index, "preventing malicious inputs", top_k=3)
vector_r4 = vector_search(owasp_vector_index, "preventing malicious inputs", model, top_k=3)
hybrid_r4 = hybrid_search(owasp_index, owasp_vector_index, "preventing malicious inputs", model, para_to_section_map, section_chunks_dict, top_k=3)

print("\n--- Text Search ---")
for i, r in enumerate(text_r4, 1):
    print(f"{i}. {r['content'][:80]}...")

print("\n--- Vector Search ---")
for i, r in enumerate(vector_r4, 1):
    print(f"{i}. [Score: {r['score']:.3f}] {r['content'][:80]}...")

print("\n--- Hybrid Search ---")
for i, r in enumerate(hybrid_r4, 1):
    print(f"{i}. [RRF: {r['score']:.3f}] {r['filename'].split('/')[-1]}")

print("\nObservation: 'malicious inputs' semantically maps to input validation content.")

In [ ]:
# Query 5: "LLM01 prevention" - Mixed query (exact code + conceptual)
# Expected: Hybrid search strength - combines both
print("=" * 80)
print("Query 5: 'LLM01 prevention' (Mixed query)")
print("Expected: Hybrid search strength - exact match + semantic")
print("=" * 80)

text_r5 = text_search(owasp_index, "LLM01 prevention", top_k=3)
vector_r5 = vector_search(owasp_vector_index, "LLM01 prevention", model, top_k=3)
hybrid_r5 = hybrid_search(owasp_index, owasp_vector_index, "LLM01 prevention", model, para_to_section_map, section_chunks_dict, top_k=3)

print("\n--- Text Search ---")
for i, r in enumerate(text_r5, 1):
    print(f"{i}. {r['filename'].split('/')[-1]}: {r['content'][:60]}...")

print("\n--- Vector Search ---")
for i, r in enumerate(vector_r5, 1):
    print(f"{i}. [Score: {r['score']:.3f}] {r['content'][:60]}...")

print("\n--- Hybrid Search ---")
for i, r in enumerate(hybrid_r5, 1):
    print(f"{i}. [RRF: {r['score']:.3f}] {r['filename'].split('/')[-1]}")

print("\nObservation: Hybrid ranks LLM01 section highest (exact 'LLM01' + semantic 'prevention').")

## Analysis Summary: Search Methods for Security Documentation

### Comparison

| Method | Best For | OWASP Examples | Limitations |
|--------|----------|----------------|-------------|
| Text Search | Exact codes, technical terms | LLM01, CVE-*, "prompt injection" | Fails on paraphrases, conceptual queries |
| Vector Search | Conceptual queries, paraphrases | "how to secure AI models", "preventing malicious inputs" | May miss exact acronyms, less precise on codes |
| Hybrid Search | Production default, mixed queries | "LLM01 prevention", any query type | Best overall, slight latency cost (runs both) |

### Key Findings

1. **Acronym Reliability** (HOMEWORK-07, HOMEWORK-10): Text search perfectly matches LLM01-10 codes via TF-IDF exact term matching. Essential for security documentation with standardized vulnerability identifiers. Vector search may rank similar-sounding content higher than exact matches.

2. **Technical Terminology** (HOMEWORK-10): Multi-word exact terms ("prompt injection", "data poisoning") benefit from text search's phrase matching and field boosting (title: 2.0x). Security teams searching for specific attack vectors get precise results.

3. **Conceptual Questions** (HOMEWORK-09): Security practitioners often search conceptually ("how to protect models from attacks"). Vector search handles semantic queries where exact terminology is unknown. Useful for threat modeling and best practices exploration.

4. **OWASP Structure Alignment** (HOMEWORK-09): Section chunking (## headers = vulnerabilities) works well with text search because each section focuses on distinct terminology (LLM01 = prompt injection, LLM04 = data poisoning). Multi-granularity approach leverages this natural structure.

5. **Production Recommendation** (HOMEWORK-09): Use hybrid search as default. Security teams need both exact code lookup (incident response, CVE tracking) and conceptual exploration (threat modeling, best practices). Multi-granularity RRF fusion provides best-of-both-worlds retrieval.

### Addressing Technical Terminology Challenges (HOMEWORK-10)

- **Vulnerability codes (LLM01-10)**: Text search excels at exact matches. Acronyms like "LLM01" have no semantic meaning to embedding models - they need lexical matching.
- **CVE format**: Text search handles exact CVE-YYYY-NNNNN format reliably when present in corpus. Consider dedicated CVE index for production systems.
- **Precise vs conceptual balance**: Hybrid search via RRF fusion addresses both needs simultaneously. Query "LLM01 prevention" gets credit from both exact "LLM01" match (text) and semantic "prevention" (vector).
- **Multi-word technical terms**: Field boosting prioritizes title matches where vulnerability names appear. "Prompt Injection" in title ranks higher than in body.

---

## Day 4: AI Agent for OWASP LLM Top 10

Apply Pydantic AI agent pattern from course/day4.ipynb to OWASP security documentation.
This section demonstrates domain-specific agent tuning:

1. **Security advisor system prompt** - Professional threat analysis, OWASP citation requirements
2. **Hybrid search integration** - Uses hybrid_search (validated optimal for OWASP in Day 3)
3. **Diverse query testing** - Direct vulnerability lookups, prevention techniques, conceptual questions

**Dependencies:** Reuses all Day 3 indexes (owasp_index, owasp_vector_index, para_to_section_map, section_chunks_dict)

In [ ]:
# Day 4: Pydantic AI imports
# pydantic-ai installed in Phase 19 (course/ and project/)
from pydantic_ai import Agent
from typing import Any

# Verify Day 3 indexes are available
# These were built in Day 3 section above - DO NOT rebuild
assert 'owasp_index' in dir(), "Run Day 3 cells first to build text index"
assert 'owasp_vector_index' in dir(), "Run Day 3 cells first to build vector index"
assert 'para_to_section_map' in dir(), "Run Day 3 cells first to build paragraph-section mapping"
assert 'section_chunks_dict' in dir(), "Run Day 3 cells first to build section chunks dict"
assert 'embedding_model' in dir(), "Run Day 3 cells first to load embedding model"

print("Day 4 imports successful")
print(f"Reusing Day 3 indexes: {len(owasp_index)} text chunks, vector index ready")
print("Ready to create OWASP security agent")